In [1]:
import gc, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from google.colab import drive, runtime

IN_COLAB = True
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026')
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA_V2, OUTPUT
from src.benchmark import analytic_benchmark
from src.helper import make_run_dir, _notebook_stem
from src.metrics import metrics, gain
from src.model3_utils import FEATURE_SETS as _ALL_FEATURE_SETS, TARGET
from src.fully_connected_colab import (
    ANN_ReLU,
    compute_batch_size,
    detect_device,
    prepare_gpu_data,
    train_one_model,
    save_colab_run,
)

Mounted at /content/drive


## Configuration

In [2]:
DATA_SETS = ['chro_A', 'chro_B', 'chro_C', 'chro_D']
NOTEBOOK_NAME = _notebook_stem()

SEED = 42
MAX_EPOCHS = 300
PATIENCE = 30
LR_PATIENCE = 8
LR_FACTOR = 0.3
WARMUP_EPOCHS = 5
NEURONS = 80
HIDDEN_LAYERS = 3
BASE_LR = 1e-3
BASE_BATCH = 4096

# Override feature sets locally if needed:
FEATURE_SETS = {k: v for k, v in _ALL_FEATURE_SETS.items() if k in ('6F_gamma+iv_lag', '8F_theta+iv_lag', '8F_rho+iv_lag')}

# Compute the union of all features across selected sets
ALL_FEATURES = list(dict.fromkeys(f for fs in FEATURE_SETS.values() for f in fs))
print(f'Feature sets: {list(FEATURE_SETS.keys())}')
print(f'ALL_FEATURES ({len(ALL_FEATURES)}): {ALL_FEATURES}')

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

Feature sets: ['6F_gamma+iv_lag', '8F_theta+iv_lag', '8F_rho+iv_lag']
ALL_FEATURES (10): ['delta', 'T', 'spy_ret', 'vix_lag', 'vix_mom_lag', 'gamma', 'iv_lag', 'vix_mom', 'theta', 'rho']


## GPU auto-detect

In [3]:
CFG = detect_device()
DEVICE = CFG['DEVICE']
AMP_DTYPE = CFG['POLICY']
USE_AMP = (AMP_DTYPE != torch.float32)

L4  |  VRAM: 24 GB  |  MAX_BATCH=32,768  |  dtype=torch.bfloat16


## Train all datasets

In [4]:
RUN_DIR = make_run_dir(NOTEBOOK_NAME)
print(f'Output directory: {RUN_DIR}')

grand_total_time = 0.0

for dataset_name in DATA_SETS:
    print(f'\n{"#" * 70}')
    print(f'#  Dataset: {dataset_name}')
    print(f'{"#" * 70}')

    # Load data
    df_train = pd.read_parquet(CLEAN_DATA_V2 / f'{dataset_name}_train_v2.parquet')
    df_val   = pd.read_parquet(CLEAN_DATA_V2 / f'{dataset_name}_val_v2.parquet')
    df_test  = pd.read_parquet(CLEAN_DATA_V2 / f'{dataset_name}_test_v2.parquet')

    print(f'Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}')

    # Analytic benchmark
    hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)

    # Prepare GPU data (all features at once, column indices used per model)
    gpu_data = prepare_gpu_data(df_train, df_val, df_test, ALL_FEATURES, TARGET, DEVICE)
    COL_IDX = gpu_data['col_idx']

    # Adaptive batch size
    BATCH_SIZE = compute_batch_size(len(df_train), CFG['MAX_BATCH'])
    INIT_LR = BASE_LR * (BATCH_SIZE / BASE_BATCH) ** 0.5
    steps_per_epoch = len(df_train) // BATCH_SIZE
    print(f'MAX_BATCH={CFG["MAX_BATCH"]:,}  adaptive BATCH_SIZE={BATCH_SIZE:,}  '
          f'INIT_LR={INIT_LR:.6f}  steps/epoch~{steps_per_epoch}')

    train_kw = dict(
        Xtr=gpu_data['Xtr'],
        Xva=gpu_data['Xva'],
        Xte=gpu_data['Xte'],
        ytr=gpu_data['ytr'],
        yva=gpu_data['yva'],
        y_test=gpu_data['y_test'],
        hw_sse=hw['sse'],
        all_feature_names=ALL_FEATURES,
        device=DEVICE,
        amp_dtype=AMP_DTYPE,
        use_amp=USE_AMP,
        nan_mask_tr=gpu_data['nan_mask_tr'],
        nan_mask_va=gpu_data['nan_mask_va'],
        nan_mask_ytr=gpu_data['nan_mask_ytr'],
        nan_mask_yva=gpu_data['nan_mask_yva'],
        seed=SEED,
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        lr_patience=LR_PATIENCE,
        lr_factor=LR_FACTOR,
        init_lr=INIT_LR,
        warmup_epochs=WARMUP_EPOCHS,
        neurons=NEURONS,
        hidden_layers=HIDDEN_LAYERS,
    )

    results_by_fs = {}

    pbar = tqdm(FEATURE_SETS.items(), desc=f'{dataset_name} feature sets', unit='set')
    for fs_name, feature_cols in pbar:
        pbar.set_postfix_str(fs_name)

        cols = [COL_IDX[f] for f in feature_cols]
        result = train_one_model(fs_name, cols, **train_kw)

        print(f'  {fs_name}: SSE={result["sse"]:.4f}  RMSE={result["rmse"]:.6f}  '
              f'Gain={result["gain_vs_hw"]:+.2f}%  epochs={result["epochs"]}  '
              f'time={result["training_time"]:.1f}s')

        results_by_fs[fs_name] = result

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ── Save results for this dataset ─────────────────────────────────
    dataset_dir = RUN_DIR / dataset_name
    summary, df_results = save_colab_run(
        dataset_dir,
        y_test=gpu_data['y_test'],
        hw=hw,
        models=results_by_fs,
    )
    print(f'\nMetrics Summary ({dataset_name}):')
    display(summary)

    # ── Dataset summary ───────────────────────────────────────────────
    ds_time = sum(r['training_time'] for r in results_by_fs.values())
    grand_total_time += ds_time
    print(f'\nFC on {dataset_name} — training time: {ds_time / 60:.1f} min')
    for fs_name, res in results_by_fs.items():
        print(f'  {fs_name}: SSE={res["sse"]:.4f}  Gain={res["gain_vs_hw"]:+.2f}%  epochs={res["epochs"]}')

    # Free memory before next dataset
    del results_by_fs, gpu_data, df_train, df_val, df_test
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Output directory: /content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026/output/6.0-fc-chro-all/03-run

######################################################################
#  Dataset: chro_A
######################################################################
Train: 2,266,676  Val: 942,247  Test: 579,718
Analytic Benchmark
SSE = 22.8915  RMSE = 0.006284
Coefficients: a = -0.142840, b = -0.082050, c = -0.058247
Data on GPU  |  VRAM used: 0.38 GB / 24 GB  |  Free: 23.3 GB
Train: 2,266,676  Val: 942,247  Test: 579,718  Features: 10
MAX_BATCH=32,768  adaptive BATCH_SIZE=32,768  INIT_LR=0.002828  steps/epoch~69


chro_A feature sets:   0%|          | 0/3 [00:00<?, ?set/s]

  6F_gamma+iv_lag: SSE=77.6954  RMSE=0.011577  Gain=-239.41%  epochs=115  time=20.9s
  8F_theta+iv_lag: SSE=94.7201  RMSE=0.012782  Gain=-313.78%  epochs=116  time=20.1s
  8F_rho+iv_lag: SSE=90.5680  RMSE=0.012499  Gain=-295.64%  epochs=147  time=25.4s

Metrics Summary (chro_A):


,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic,Gain_Incremental
0,Analytic,22.891512,0.000039,0.006284,0.003471,0.000647,0.002053,0.205463,None,None,None
1,6F_gamma+iv_lag,77.695351,0.000134,0.011577,0.007890,-0.001475,0.005605,-1.696714,20.9s,-239.41%,None
2,8F_theta+iv_lag,94.720108,0.000163,0.012782,0.008444,-0.000814,0.005960,-2.287623,20.1s,-313.78%,-21.91%
3,8F_rho+iv_lag,90.568047,0.000156,0.012499,0.007895,0.000739,0.004868,-2.143510,25.4s,-295.64%,4.38%
4,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,66.5s,None,None



FC on chro_A — training time: 1.1 min
  6F_gamma+iv_lag: SSE=77.6954  Gain=-239.41%  epochs=115
  8F_theta+iv_lag: SSE=94.7201  Gain=-313.78%  epochs=116
  8F_rho+iv_lag: SSE=90.5680  Gain=-295.64%  epochs=147

######################################################################
#  Dataset: chro_B
######################################################################
Train: 744,477  Val: 331,626  Test: 225,573
Analytic Benchmark
SSE = 39.5613  RMSE = 0.013243
Coefficients: a = -0.082956, b = -0.181799, c = -0.174770
Data on GPU  |  VRAM used: 0.55 GB / 24 GB  |  Free: 23.1 GB
Train: 744,477  Val: 331,626  Test: 225,573  Features: 10
MAX_BATCH=32,768  adaptive BATCH_SIZE=8,192  INIT_LR=0.001414  steps/epoch~90


...


FC on chro_C — training time: 1.2 min
  6F_gamma+iv_lag: SSE=36.6959  Gain=-250.64%  epochs=132
  8F_theta+iv_lag: SSE=11.5187  Gain=-10.06%  epochs=209
  8F_rho+iv_lag: SSE=25.0630  Gain=-139.49%  epochs=222

######################################################################
#  Dataset: chro_D
######################################################################
Train: 790,689  Val: 265,453  Test: 148,363
Analytic Benchmark
SSE = 7.3640  RMSE = 0.007045
Coefficients: a = -0.167575, b = -0.000015, c = 0.033513
Data on GPU  |  VRAM used: 0.47 GB / 24 GB  |  Free: 23.2 GB
Train: 790,689  Val: 265,453  Test: 148,363  Features: 10
MAX_BATCH=32,768  adaptive BATCH_SIZE=8,192  INIT_LR=0.001414  steps/epoch~96


chro_D feature sets:   0%|          | 0/3 [00:00<?, ?set/s]

  6F_gamma+iv_lag: SSE=8.3624  RMSE=0.007508  Gain=-13.56%  epochs=175  time=40.9s
  8F_theta+iv_lag: SSE=9.8574  RMSE=0.008151  Gain=-33.86%  epochs=185  time=43.0s
  8F_rho+iv_lag: SSE=9.0778  RMSE=0.007822  Gain=-23.27%  epochs=186  time=43.8s

Metrics Summary (chro_D):


,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic,Gain_Incremental
0,Analytic,7.364048,0.000050,0.007045,0.003278,0.000561,0.001530,0.081872,None,None,None
1,6F_gamma+iv_lag,8.362355,0.000056,0.007508,0.004375,-0.000171,0.002805,-0.042595,40.9s,-13.56%,None
2,8F_theta+iv_lag,9.857394,0.000066,0.008151,0.004584,-0.001466,0.002580,-0.228992,43.0s,-33.86%,-17.88%
3,8F_rho+iv_lag,9.077817,0.000061,0.007822,0.004352,-0.001162,0.002551,-0.131796,43.8s,-23.27%,7.91%
4,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,127.7s,None,None



FC on chro_D — training time: 2.1 min
  6F_gamma+iv_lag: SSE=8.3624  Gain=-13.56%  epochs=175
  8F_theta+iv_lag: SSE=9.8574  Gain=-33.86%  epochs=185
  8F_rho+iv_lag: SSE=9.0778  Gain=-23.27%  epochs=186


## Grand Summary

In [5]:
print(f'\n{"=" * 70}')
print(f'FC on all datasets — grand total training time: {grand_total_time / 60:.1f} min')
print(f'Results saved to: {RUN_DIR}')
for ds in DATA_SETS:
    print(f'  {ds}: {RUN_DIR / ds}')


FC on all datasets — grand total training time: 6.1 min
Results saved to: /content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026/output/6.0-fc-chro-all/03-run
  chro_A: /content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026/output/6.0-fc-chro-all/03-run/chro_A
  chro_B: /content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026/output/6.0-fc-chro-all/03-run/chro_B
  chro_C: /content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026/output/6.0-fc-chro-all/03-run/chro_C
  chro_D: /content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026/output/6.0-fc-chro-all/03-run/chro_D


## Cleanup

In [6]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'Final VRAM: {(total - free) / 1e9:.2f} GB / {total / 1e9:.0f} GB')

print(f'Grand total training time: {grand_total_time / 60:.1f} min')

if IN_COLAB:
    runtime.unassign()

Final VRAM: 0.47 GB / 24 GB
Grand total training time: 6.1 min
